#Rag Pipelines-Data ingestions to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\Acer\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")


Found 1 PDF files to process

Processing: Resume_of_Yogesh_Kumar.pdf
  ✓ Loaded 2 pages

Total documents loaded: 2


In [3]:

### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks

Split 2 documents into 8 chunks

Example chunk:
Content: Yogesh  Kumar New  Delhi,  India   |   yogesh.bhandari285@gmail.com  |   +91-8384855717 LinkedIn   |   Portfolio   |   GitHub  |   LeetCode  
PROFESSIONAL  SUMMARY 
Full  Stack  Engineer  with  4.5+  ...
Metadata: {'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Resume of Yogesh Kumar', 'source': '..\\data\\pdf_files\\Resume_of_Yogesh_Kumar.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Resume_of_Yogesh_Kumar.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Resume of Yogesh Kumar', 'source': '..\\data\\pdf_files\\Resume_of_Yogesh_Kumar.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Resume_of_Yogesh_Kumar.pdf', 'file_type': 'pdf'}, page_content='Yogesh  Kumar New  Delhi,  India   |   yogesh.bhandari285@gmail.com  |   +91-8384855717 LinkedIn   |   Portfolio   |   GitHub  |   LeetCode  \nPROFESSIONAL  SUMMARY \nFull  Stack  Engineer  with  4.5+  years  of  experience  building  scalable  web  applications,  secure  REST  APIs,  and  AI-powered  systems  \nusing\n \nLaravel,\n \nDjango,\n \nFastAPI,\n \nand\n \nReact.\n \nExperienced\n \nin\n \nmulti-agent\n \nLLM\n \narchitectures\n \n(LangGraph,\n \nLangChain,\n \nRAG),\n \nreal-time\n \nsystems\n \n(WebSockets,\n \nKafka),\n \nand\n \ncloud-native\n \ndeployments\n \n(Docker,\n \nAWS,\n \nCI/CD).\n \nProven\n \nability\n \nto\n \ndesign\n \nproducti

Embedding and VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 787.85it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [8]:
### convert the text to embeddings

texts=[doc.page_content for doc in chunks]

## generate the embeddings for the chunks
embeddings=embedding_manager.generate_embeddings(texts)

## store the embeddings in the vector store

vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.76it/s]

Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 8


Retriever pipeline from vectorstore

In [9]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [10]:
rag_retriever.retrieve('PROFESSIONAL SUMMARY')

Retrieving documents for query: 'PROFESSIONAL SUMMARY'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.65it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0eed515f_2',
  'content': '20%\n \nincrease\n \nin\n \nmonthly\n \nactive\n \nusers. •  Payment  Systems:  Implemented  secure  wallet,  transaction,  and  mandate  management  modules;  integrated  multiple  payment  \ngateways\n \n(PayU,\n \nUPI\n \nAutopay). \n•  Scalable  API  Design:  Created  modular,  RESTful  APIs  following  clean  architecture  principles;  decreased  average  API  response  \ntime\n \nby\n \n25%\n \nand\n \nenhanced\n \nbackend\n \nsecurity.  \nSenior  Lead  Developer Lavessta  Enterprise  Pvt  Ltd \nApr  2022  –  Mar  2024 Noida •  AI-Powered  Features:  Implemented  ML/AI  workflows  for  data  extraction  and  analysis;  increased  project  revenue  by  15%. •  Scalable  REST  APIs:  Engineered  fault-resistant  RESTful  web  services  supporting  1000+  concurrent  users. •  Team  Lead:  Oversaw  backend  development  and  coordinated  delivery  with  team  members;  reduced  production-level  user-reported  \nbugs\n \nby\n \n35%\n \nacross\

Integrations